In [0]:
# connect to data lake
%run ../01_setup/storage_config.ipynb

In [0]:
# Manually add schema to ensure fields are parsed correctly
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType

raw_schema = StructType(fields = [
    StructField("Service:RDT-ID", StringType(), False), #service id cannot be null
    StructField("Service:Date", StringType(), True),
    StructField("Service:Type", StringType(), True),
    StructField("Service:Company", StringType(), True),
    StructField("Service:Train number", StringType(), True),
    StructField("Service:Completely cancelled", BooleanType(), True),
    StructField("Service:Partly cancelled", BooleanType(), True),
    StructField("Service: Maximum delay", IntegerType(), True),
    StructField("Stop:RDT-ID", StringType(), True),
    StructField("Stop:Station code", StringType(), True),
    StructField("Stop:Station name", StringType(), True),
    StructField("Stop:Arrival time", StringType(), True),
    StructField("Stop:Arrival delay", IntegerType(), True),
    StructField("Stop:Arrival cancelled", BooleanType(), True),
    StructField("Stop:Departure time", StringType(), True),
    StructField("Stop:Departure delay", IntegerType(), True),
    StructField("Stop:Departure cancelled", BooleanType(), True),
    StructField("Stop:Platform change", BooleanType(), True),
    StructField("Stop:Planned platform", StringType(), True),
    StructField("Stop:Actual platform", StringType(), True)
    ])

### Step 1 - fetch data in csv format

In [0]:
# read the csv file
raw_df = spark.read.schema(raw_schema).csv(source_file_path, header=True)

In [0]:
# quick glance at the dataframe
raw_df.limit(5)

### Step 2 - rename the columns for consistency

In [0]:
from pyspark.sql import DataFrame
# TO DO - add test to function
def clean_column_names(df:DataFrame) -> DataFrame: 
  """
  Cleans column names by removing spaces and colons and replacing them with underscores.
  
  Args:
    df: A Spark DataFrame.
  
  Returns:
    A Spark DataFrame with cleaned column names.
  """
  column_list = df.columns
  for column in column_list:
    new_column_name = column. \
      replace(":", " ").replace(" ", "_").replace("-", "_").replace("__", "_") \
      .lower() # convert to snake case
    df = df.withColumnRenamed(column, new_column_name)
  return df

In [0]:
renamed_df = clean_column_names(raw_df)


### Step 3 - Add ingestion time stamp

In [0]:
from pyspark.sql.functions import lit, current_timestamp
final_df = renamed_df.withColumn("file_source", lit("csv")) \
    .withColumn("ingestion_date", current_timestamp()) 

### Step 4 - write data to delta lake

In [0]:
final_df.write.mode("overwrite").format("delta").save(raw_folder_path)

### Step 5 - create databases and tables

In [0]:
database_name = "ns_train_raw"
tabel_name = "services"

# db creates delta table by default
create_db_sql = f"""
CREATE DATABASE IF NOT EXISTS {database_name}
"""

spark.sql(create_db_sql)
spark.sql(f'USE DATABASE {database_name}')

In [0]:
create_table_sql = f"""
CREATE TABLE IF NOT EXISTS {database_name}.{tabel_name}
LOCATION "{raw_folder_path}"
"""
spark.sql(create_table_sql)

In [0]:
%sql
SELECT * FROM services
LIMIT 5;